[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-04-memory-scoring-relevance-recency-importance.ipynb)

# Advanced Discussion 4 — memory scoring

Retrieval over a synthetic memory with changing facts: relevance only versus recency and importance.

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Downloads a small embedding model (about 90 MB).

In [ ]:
%pip install -q sentence-transformers

In [ ]:
import numpy as np, math
from sentence_transformers import SentenceTransformer
rng = np.random.RandomState(0)

## 1. a synthetic long-running assistant memory: preferences that CHANGE over time, buried in chatter

In [ ]:
TOPICS = [  # (query, old fact, new fact)
 ("What format does the user want weekly reports in?", "The user wants weekly reports as a PDF.", "The user now wants weekly reports as a spreadsheet."),
 ("Which currency should quotes use?", "The user asked for all quotes in US dollars.", "The user switched quotes to euros."),
 ("What time should the daily summary be sent?", "Send the daily summary at 8 am.", "Send the daily summary at 6 pm instead."),
 ("Who approves purchases over the limit?", "Purchases over the limit are approved by Dana.", "Purchases over the limit are now approved by Marcus."),
 ("Which warehouse ships the large orders?", "Large orders ship from the Rotterdam warehouse.", "Large orders now ship from the Antwerp warehouse."),
 ("What language should customer emails be in?", "Customer emails should be written in English.", "Customer emails should now be written in German."),
 ("How should delays be reported?", "Report delays by email to the whole team.", "Report delays only in the ops chat channel."),
 ("Which carrier handles refrigerated freight?", "Refrigerated freight goes with ColdChain.", "Refrigerated freight now goes with FrostLine."),
 ("What is the escalation contact for outages?", "For outages, call Priya first.", "For outages, call the on-call phone first."),
 ("What units should distances use?", "Show distances in kilometres.", "Show distances in miles."),
 ("Which dashboard should the user check in the morning?", "The user checks the fleet dashboard each morning.", "The user now checks the cost dashboard each morning."),
 ("What is the preferred meeting length?", "The user prefers 45-minute meetings.", "The user prefers 25-minute meetings."),
]
FILLER = ["It looks like rain tomorrow.", "The printer on the second floor is jammed again.", "Lunch was moved to the terrace today.", "Thanks, that was helpful.",
          "Please remind me about the parking permit.", "The quarterly offsite is in spring.", "I updated my profile photo.", "Sorry, I was in a meeting.",
          "The coffee machine needs descaling.", "Ok, sounds good.", "Can you also check the holiday calendar?", "The intern starts on Monday.",
          "I will be travelling next week.", "The elevator is out of service.", "Noted, thanks for the update.", "Let me think about that and get back to you."]
memories = []   # (text, day, importance)
for q, old, new in TOPICS:
    d_old = rng.randint(1, 40); d_new = rng.randint(60, 120); memories += [(old, d_old, 0.9), (new, d_new, 0.9)]
for _ in range(300): memories.append((FILLER[rng.randint(len(FILLER))] + f" ({rng.randint(1000)})", rng.randint(1, 121), rng.choice([0.1, 0.2, 0.3], p=[.6, .3, .1])))
texts = [m[0] for m in memories]; days = np.array([m[1] for m in memories], float); imp = np.array([m[2] for m in memories])
now = 121.0
enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
E = enc.encode(texts, normalize_embeddings=True); Qv = enc.encode([t[0] for t in TOPICS], normalize_embeddings=True)
new_idx = [texts.index(t[2]) for t in TOPICS]; old_idx = [texts.index(t[1]) for t in TOPICS]
print("memory store: %d entries (%d preference facts, %d chatter lines); %d questions whose correct answer is the NEWER fact" % (len(texts), 2 * len(TOPICS), 300, len(TOPICS)))

## 2. scoring functions: relevance only vs relevance + recency + importance (Generative-Agents-style)

In [ ]:
def evaluate(score_fn, k_list=(1, 3)):
    out = {k: 0 for k in k_list}; stale_at1 = 0
    for qi, (ni, oi) in enumerate(zip(new_idx, old_idx)):
        sc = score_fn(E @ Qv[qi]); top = np.argsort(-sc)
        for k in k_list: out[k] += ni in top[:k]
        stale_at1 += top[0] == oi
    n = len(TOPICS); return {k: v / n for k, v in out.items()}, stale_at1 / n
def relevance_only(sim): return sim
def with_recency(half_life):
    return lambda sim: sim + 0.5 * np.exp(-(now - days) * math.log(2) / half_life)
def full(half_life, w_imp):
    return lambda sim: sim + 0.5 * np.exp(-(now - days) * math.log(2) / half_life) + w_imp * imp
print("\nscoring rule                                     newest fact in top-1   in top-3   stale (old) fact ranked first")
for name, fn in (("relevance only", relevance_only), ("relevance + recency (half-life 90 days)", with_recency(90)), ("relevance + recency (half-life 30 days)", with_recency(30)),
                 ("relevance + recency (30 d) + importance", full(30, 0.3)), ("recency only", lambda sim: np.exp(-(now - days) / 30))):
    (r, stale) = evaluate(fn); print("%-46s  %8.2f            %6.2f     %6.2f" % (name, r[1], r[3], stale))

## 3. compaction arithmetic: what does it cost to keep the raw history in the prompt?

In [ ]:
avg_tokens_per_memory = 18
print("\nkeeping all %d entries in the prompt costs ~%d tokens per request; a top-5 retrieval costs ~%d tokens (%.0f%% of the raw history)" % (len(texts), len(texts) * avg_tokens_per_memory, 5 * avg_tokens_per_memory, 100 * 5 / len(texts)))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.